In [ ]:
# ================================
# 07-test-set-evaluation.ipynb
# Evaluates all 108 previously trained PEFT adapters on the held-out test sets.
# ================================

# ------------------------------
# 1. Environment Setup
# ------------------------------
!pip uninstall -y torchao
!pip install -q peft --no-deps
!pip install -q accelerate

import torch, os, time, json, gc, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

# ------------------------------
# 2. Configuration & Paths
# ------------------------------
MODEL_NAME = "xlm-roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 64  # Fast evaluation batch size

DATA_ROOT = "/kaggle/input/notebooks/venkatkolluu/02-data-preprocessingv2/data/processed"
PRIMARY_RESULTS_ROOT = "/kaggle/input/notebooks/venkatkolluu/05-full-experiment-sweep"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the experiment list (108 runs)
sweep_df = pd.read_csv(f"{PRIMARY_RESULTS_ROOT}/experiment_results.csv")
print(f"Loaded {len(sweep_df)} runs from {PRIMARY_RESULTS_ROOT}/experiment_results.csv")

# ------------------------------
# 3. Load Held-Out Test Data
# ------------------------------
def build_test_loader(language):
    test_df = pd.read_parquet(f"{DATA_ROOT}/{language}/test.parquet")
    def tokenize(batch):
        return tokenizer(
            batch["premise"], batch["hypothesis"],
            truncation=True, padding="max_length", max_length=MAX_LENGTH
        )
    ds = Dataset.from_pandas(test_df).rename_column("label", "labels").map(tokenize, batched=True)
    keep = ["input_ids", "attention_mask", "labels"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    ds.set_format("torch")
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

loaders = {
    "hi": build_test_loader("hi"),
    "te": build_test_loader("te")
}
print(f"Loaded test sets: Hindi ({len(loaders['hi'].dataset)} samples), Telugu ({len(loaders['te'].dataset)} samples)")

# ------------------------------
# 4. Helper: Robust Model Loader (Handles Classifier State Dict)
# ------------------------------
def load_evaluated_model(adapter_path):
    """
    Loads base model, PEFT adapter, and explicitly restores the saved classifier_head.pt
    exactly matching how notebook 05 saved them.
    """
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).cuda()
    
    model = PeftModel.from_pretrained(base_model, adapter_path)
    
    classifier_path = os.path.join(adapter_path, "classifier_head.pt")
    if os.path.exists(classifier_path):
        classifier_state = torch.load(classifier_path, map_location="cuda", weights_only=True)
        model.load_state_dict(classifier_state, strict=False)
    
    model.eval()
    return model

# ------------------------------
# 5. Evaluation Loop (Resumable)
# ------------------------------
results_file = "/kaggle/working/test_results.csv"

completed_keys = set()
if os.path.exists(results_file):
    try:
        existing_df = pd.read_csv(results_file)
        for _, row in existing_df.iterrows():
            completed_keys.add((row["method"], row["language"], int(row["budget"]), int(row["seed"])))
        print(f"Found {len(completed_keys)} previously completed evaluations. Resuming...")
    except Exception as e:
        print(f"Could not read existing results: {e}")

test_results = []
total_runs = len(sweep_df)

for idx, row in sweep_df.iterrows():
    method = str(row['method'])
    lang = str(row['language'])
    budget = int(row['budget'])
    seed = int(row['seed'])
    
    key = (method, lang, budget, seed)
    if key in completed_keys:
        print(f"[{idx+1}/{total_runs}] SKIP (Already Done): {method} | {lang} | budget={budget} | seed={seed}")
        continue

    adapter_path = f"{PRIMARY_RESULTS_ROOT}/adapters/{method}/{lang}/budget{budget}_seed{seed}"
    print(f"[{idx+1}/{total_runs}] Evaluating {method} | {lang} | budget={budget} | seed={seed}")

    try:
        model = load_evaluated_model(adapter_path)
        
        preds, labels = [], []
        with torch.no_grad():
            for batch in loaders[lang]:
                batch = {k: v.cuda() for k, v in batch.items()}
                outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
                preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
                labels.extend(batch["labels"].cpu().numpy())
                
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average="macro")
        
        res = {
            "method": method, "language": lang, "budget": budget, "seed": seed,
            "test_accuracy": round(acc, 6), "test_macro_f1": round(f1, 6)
        }
        test_results.append(res)
        
        pd.DataFrame([res]).to_csv(
            results_file, mode='a', header=not os.path.exists(results_file), index=False
        )
        print(f"   -> Test Acc: {acc:.4f} | Test Macro-F1: {f1:.4f}")
        
    except Exception as e:
        print(f"   -> ERROR: {e}")
    finally:
        if 'model' in locals():
            del model
        gc.collect()
        torch.cuda.empty_cache()

print("\n✅ Test set evaluation complete!")
if os.path.exists(results_file):
    final_df = pd.read_csv(results_file)
    print(f"Total evaluated runs saved: {len(final_df)}/{total_runs}")

